[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thebnbrkr/marv-audio/blob/main/notebooks/02_reproduce_experiments.ipynb)

# Reproduce E1, E2 and E3

Reruns the experiments from `PREDICTIONS.md` on whisper-tiny (20 LibriSpeech
clips) and plots them. About 4 minutes on a free Colab CPU.

- **E1**: how much of each token comes from listening vs guessing.
- **E2**: which writes carry the audio's effect to the output.
- **E3**: the audio's effect split exactly into parts, in matched units.

In [ ]:
!git clone -q https://github.com/thebnbrkr/marv-audio
%cd marv-audio
!pip install -q -e ".[experiments]"

In [ ]:
!python scripts/listen_vs_guess.py --out results/e1_rerun.json | tail -45

In [ ]:
!python scripts/audio_routes.py --out results/e2_rerun.json | tail -30

In [ ]:
!python scripts/matched_split.py --out results/e3_rerun.json | tail -25

In [ ]:
import json
import matplotlib.pyplot as plt

e1 = json.load(open("results/e1_rerun.json"))["summary"]
e2 = json.load(open("results/e2_rerun.json"))["summary"]
e3 = json.load(open("results/e3_rerun.json"))["summary"]

fig, (a, b, c, d) = plt.subplots(1, 4, figsize=(18, 4))
kinds = list(e1["by_kind"])
a.bar(kinds, [e1["by_kind"][k]["median_total_effect_nats"] for k in kinds])
a.set_title("E1: audio effect by token kind"); a.set_ylabel("median nats lost on silence")

routes = e2["share_of_T_summed"]
names = ["direct", "via MLP", "via self-attn"]
for i, order in enumerate(["mlp_first", "self_first"]):
    vals = [routes["direct"], routes[order]["via_mlp"], routes[order]["via_self"]]
    b.bar([x + i * 0.4 for x in range(3)], vals, width=0.4, label=order.replace("_", " "))
b.set_xticks([x + 0.2 for x in range(3)], names); b.legend(); b.set_title("E2: route of the audio effect")

c.bar([f"layer {i}" for i in range(len(e2["median_loss_share_by_layer"]))], e2["median_loss_share_by_layer"])
c.set_title("E2: share lost removing one layer's cross-attn")
parts = ["cross_attn", "mlp", "self_attn", "embed"]
d.bar(parts, [e3["share_of_delta"][p] for p in parts])
d.axhline(0, color="black", lw=0.8)
d.set_title("E3: share of the audio's effect (matched units)")
plt.tight_layout(); plt.show()

Compare with the recorded outcomes in `PREDICTIONS.md`: the model revision is
pinned, so the numbers should match to several decimals (CPU vs GPU and library
versions can move the last digits).